# Daily Data Pipeline — Fund Risk

This notebook implements the daily morning data validation and enrichment 
workflow that runs before any risk calculations begin. The automated 
pipeline has already loaded the previous day's positions into the database 
overnight. What follows is the validation, enrichment and exception review 
that must complete before risk metrics are computed and reported.


**Context**

Fund administrator position files arrive daily, typically between 7am and 
9am, reflecting the previous day's closing positions and prices. Before 
these feed into VaR models and stress tests, three questions must be 
answered:

1. **Are the prices correct?** Fund administrator pricing can differ from 
   Bloomberg, particularly for bonds and illiquid instruments where 
   model-based or stale prices are common.

2. **Are there new instruments?** Any ISIN that was not in yesterday's 
   portfolio requires reference data from Bloomberg before it can be 
   risk-managed correctly.

3. **Is the market context consistent with the risk numbers?** A VaR 
   figure means something different when the VIX is at 30 versus 15. 
   Market context is essential for interpreting risk metrics correctly.

The workflow below simulates this process for the **AIFM Hedge Fund** on 
**13 May 2026**, using a mock Bloomberg client that mirrors the real 
`blpapi` interface. Switching to production Bloomberg requires changing 
one import line.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlalchemy as sa
from sqlalchemy import text
import warnings
warnings.filterwarnings('ignore')

# switch to real Bloomberg in one line
from src.mock_bloomberg import MockBloomberg as Bloomberg
from src.database import get_engine, query_positions, query_nav_history
from src.enrichment import enrich_positions, get_risk_ready_df

# ----------------------------------------------------------------
# Configuration
# ----------------------------------------------------------------
FUND_ID    = 'AIFM_HedgeFund'
TODAY      = '2026-05-13'
YESTERDAY  = '2026-05-12'
ENGINE     = get_engine()
BBG        = Bloomberg()

plt.rcParams.update({
    'figure.facecolor' : '#0f0f0f',
    'axes.facecolor'   : '#1a1a1a',
    'axes.edgecolor'   : '#333333',
    'axes.labelcolor'  : '#cccccc',
    'xtick.color'      : '#888888',
    'ytick.color'      : '#888888',
    'text.color'       : '#cccccc',
    'grid.color'       : '#2a2a2a',
    'grid.linestyle'   : '--',
    'font.family'      : 'monospace',
    'figure.dpi'       : 120,
})

ACCENT  = '#00ff9f'
ACCENT2 = '#ff6b35'
ACCENT3 = '#4fc3f7'

print(f'Fund    : {FUND_ID}')
print(f'Date    : {TODAY}')
print(f'Engine  : {ENGINE.url}')

MockBloomberg: connected (simulation mode)
Swap import to RealBloomberg for production use.
Fund    : AIFM_HedgeFund
Date    : 2026-05-13
Engine  : sqlite:////Users/mrspatbile/Documents/coding/manco-risk-mngmt/data/risk_management.db


## 1. Load Today's Positions

Load the latest positions from the database and compare with yesterday
to identify what changed overnight: new instruments, removed instruments,
and significant position size changes.

In [2]:
# ----------------------------------------------------------------
# Ensure database and position files exist
# ----------------------------------------------------------------
from pathlib import Path
import subprocess

ROOT_DIR = Path('..').resolve()
DB_PATH  = ROOT_DIR / 'data' / 'risk_management.db'
DATA_DIR = ROOT_DIR / 'data'

if not DB_PATH.exists():
    print('Database not found. Generating position files and database...')
    subprocess.run(['python3', str(ROOT_DIR / 'src' / 'generate_positions.py')],
                   check=True)
    subprocess.run(['python3', str(ROOT_DIR / 'src' / 'database.py')],
                   check=True)
    print('Done.')
else:
    print(f'Database found: {DB_PATH}')
    print(f'Size: {DB_PATH.stat().st_size / 1024:.1f} KB')

Database found: /Users/mrspatbile/Documents/coding/manco-risk-mngmt/data/risk_management.db
Size: 1872.0 KB


In [5]:
# load today and yesterday
today     = query_positions(ENGINE, FUND_ID, TODAY)
yesterday = query_positions(ENGINE, FUND_ID, YESTERDAY)

print(f'Today    : {len(today)} positions')
print(f'Yesterday: {len(yesterday)} positions')

# what changed overnight?
new_isins     = set(today['isin']) - set(yesterday['isin'])
removed_isins = set(yesterday['isin']) - set(today['isin'])

print(f'\nNew instruments     : {len(new_isins)}')
print(f'Removed instruments : {len(removed_isins)}')

if new_isins:
    print(f'\nNew ISINs:')
    for isin in new_isins:
        name = today[today['isin'] == isin]['instrument_name'].values[0]
        print(f'  {isin} : {name}')

if removed_isins:
    print(f'\nRemoved ISINs:')
    for isin in removed_isins:
        name = yesterday[
            yesterday['isin'] == isin]['instrument_name'].values[0]
        print(f'  {isin} : {name}')

# portfolio snapshot
print(f'\n--- Portfolio snapshot ({TODAY}) ---')
nav = today['market_value_eur'].sum()
print(f'NAV                 : EUR {nav:,.0f}')
print(f'Long exposure       : EUR {today[today["market_value_eur"] > 0]["market_value_eur"].sum():,.0f}')
print(f'Short exposure      : EUR {today[today["market_value_eur"] < 0]["market_value_eur"].sum():,.0f}')
print(f'Gross exposure      : EUR {today["market_value_eur"].abs().sum():,.0f}')
print(f'\nAsset class breakdown:')
breakdown = today.groupby('asset_class')['market_value_eur'].sum().sort_values(ascending=False)
for ac, mv in breakdown.items():
    pct = mv / nav * 100
    print(f'  {ac:15s}: EUR {mv:>15,.0f}  ({pct:+.1f}%)')# load today and yesterday
today     = query_positions(ENGINE, FUND_ID, TODAY)
yesterday = query_positions(ENGINE, FUND_ID, YESTERDAY)

print(f'Today    : {len(today)} positions')
print(f'Yesterday: {len(yesterday)} positions')


Today    : 14 positions
Yesterday: 14 positions

New instruments     : 0
Removed instruments : 0

--- Portfolio snapshot (2026-05-13) ---
NAV                 : EUR 94,623,895
Long exposure       : EUR 105,310,570
Short exposure      : EUR -10,686,675
Gross exposure      : EUR 115,997,245

Asset class breakdown:
  Equity         : EUR      60,120,885  (+63.5%)
  FX             : EUR      15,922,100  (+16.8%)
  Cash           : EUR      10,000,000  (+10.6%)
  Bond           : EUR       8,983,190  (+9.5%)
  Derivative     : EUR        -402,280  (-0.4%)
Today    : 14 positions
Yesterday: 14 positions


## 2. Price Validation

Fund administrator prices can differ from Bloomberg, particularly for
bonds where model-based or matrix pricing is common, and for instruments
that have not traded recently. Any discrepancy above the tolerance
threshold requires investigation before risk metrics are computed.

Tolerance thresholds:
- **equities and FX**: 0.5%
- **bonds**: 0.25% — small price differences translate into significant
  P&L on large notionals once duration is applied

In [ ]:
# pull Bloomberg prices for all liquid instruments
liquid = today[today['bloomberg_ticker'].notna()].copy()
tickers = liquid['bloomberg_ticker'].tolist()

bbg_prices = BBG.bdp(tickers, 'PX_LAST').reset_index()
bbg_prices = bbg_prices.rename(columns={
    'security': 'bloomberg_ticker',
    'PX_LAST' : 'bbg_price'
})

# merge fund admin price vs Bloomberg price
validation = liquid[['instrument_name', 'asset_class',
                      'bloomberg_ticker', 'price']].merge(
    bbg_prices, on='bloomberg_ticker', how='left'
)
validation = validation.rename(columns={'price': 'fund_admin_price'})

# compute discrepancy
validation['diff_pct'] = (
    (validation['fund_admin_price'] - validation['bbg_price']) /
    validation['bbg_price'] * 100
).round(4)

# flag by asset class tolerance
def flag_discrepancy(row):
    if row['asset_class'] == 'Derivative':
        return 'SKIP - option pricing'
    if pd.isna(row['bbg_price']):
        return 'NO BBG PRICE'
    tolerance = 0.25 if row['asset_class'] == 'Bond' else 0.50
    if abs(row['diff_pct']) > tolerance:
        return f'FLAG (>{tolerance}%)'
    return 'OK'

validation['status'] = validation.apply(flag_discrepancy, axis=1)

# display
print('--- Price validation report ---')
print(f'{"Instrument":<30} {"Asset Class":<12} {"Fund Admin":>12} {"Bloomberg":>12} {"Diff%":>8} {"Status"}')
print('-' * 90)
for _, row in validation.iterrows():
    bbg_str = f'{row["bbg_price"]:>12.4f}' if not pd.isna(
        row['bbg_price']) else f'{"N/A":>12}'
    print(f'{row["instrument_name"]:<30} {row["asset_class"]:<12} '
          f'{row["fund_admin_price"]:>12.4f} {bbg_str} '
          f'{row["diff_pct"]:>8.4f} {row["status"]}')

flagged = validation[validation['status'].str.startswith('FLAG')]
print(f'\nFlagged: {len(flagged)} / {len(validation)} positions')
if len(flagged) > 0:
    print('Requires investigation before risk metrics are computed.')

--- Price validation report ---
Instrument                     Asset Class    Fund Admin    Bloomberg    Diff% Status
------------------------------------------------------------------------------------------
SPDR S&P 500 ETF               Equity           523.4200     523.4200   0.0000 OK
Apple Inc                      Equity           211.4500     211.4500   0.0000 OK
Microsoft Corp                 Equity           415.3200     415.3200   0.0000 OK
JPMorgan Chase                 Equity           248.7300     248.7300   0.0000 OK
Euro Stoxx 50 Future           Equity          5124.8700    5124.8700   0.0000 OK
Tesla Inc (Short)              Equity           175.3400          N/A      nan NO BBG PRICE
Nvidia Corp (Short)            Equity           892.5400          N/A      nan NO BBG PRICE
T 2.875 05/15/28               Bond              96.4200      96.4200   0.0000 OK
DBR 0 08/15/29                 Bond              90.8700      90.8700   0.0000 OK
LVMH 3.5 06/15/31              Bo